In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu

In [ ]:
data_with_groups = pd.read_csv('../data/data_with_groups.csv')
data_invest = pd.read_csv('../data/investigator_nacc72.csv')
data_biom = pd.read_csv('../data/bio_marker.csv')

In [ ]:
display(data_with_groups.head())
set(data_with_groups['Group'])
print(set(data_with_groups['Group']))

In [ ]:
data_NL_MCI = data_with_groups[data_with_groups['Group'] == 'NL to MCI' ]
data_NL_iMCI = data_with_groups[data_with_groups['Group'] == 'NL to I.,nMCI']
data_NL_NL = data_with_groups[data_with_groups['Group'] == 'NL to NL']
print(len(data_NL_MCI))
print(len(data_NL_iMCI))
print(len(data_NL_NL))

In [ ]:
data_invest_with_groups = data_invest.merge(
    data_with_groups[['NACCID', 'Group']],
    on='NACCID',
    how='left'
)

In [ ]:
data_invest_with_groups.head()
print(data_invest_with_groups['Group'].isna().count())

In [ ]:
print(set(data_invest_with_groups.columns))

In [ ]:
results = []
for col in set(data_invest_with_groups.columns):
    if not pd.api.types.is_numeric_dtype(data_invest_with_groups[col]):
        continue

    group_stable = data_invest_with_groups[data_invest_with_groups['Group'] == 'NL to NL'][col].dropna()
    group_convert = data_invest_with_groups[data_invest_with_groups['Group'] == 'NL to MCI'][col].dropna()
    stat, p = mannwhitneyu(group_stable, group_convert, alternative='two-sided')
    if p < 0.005:
        print(f"{col}:p={p:.4f}")
    n1, n2 = len(group_stable), len(group_convert)
    r = 1 - (2 * stat) / (n1 * n2)

    print(f"{col}: p={p:.2e}, effect size r={r:.3f}")
    results.append({
        'feature': col,
        'p_value': p,
        'effect_size_r': r,
        'abs_effect_size': abs(r)
    })

results_df = pd.DataFrame(results)

leakage_vars = [
    'NACCMCII', 'NACCMCIL', 'NACCMCIA', 'NACCMCIE', 'NACCTMCI',
    'NACCNORM', 'NORMCOG', 'DEMENTED', 'NACCALZD', 'NACCALZP',
    'NACCUDSD', 'COGMODE', 'COGSTAT', 'COURSE', 'DECIN',
    'FRSTCHG', 'NACCCOGF', 'NACCETPR', 'NACCLBDP', 'NACKLBDE',
    'DEPIF', 'HUNTIF', 'PSPIF', 'DYSILLIF', 'CORTIF',
    'COGOTHIF', 'COGOTH2F', 'COGOTH3F', 'FTLDNOIF', 'MSAIF',
    'HIVIF', 'EPILEPIF', 'BRNINJIF', 'PRIONIF', 'NEOPIF',
    'ALCDEMIF', 'BIPOLDIF', 'SCHIZOIF', 'DELIRIF', 'FTLDMOIF',
    'OTHCOGIF', 'IMPSUBIF', 'ANXIETIF', 'CVDIF', 'ESSTREIF',
    'PTSDDXIF', 'BIRTHYR', 'DECAGE', 'NACCDAGE', 'NACCYOD',
]

results_clean = results_df[~results_df['feature'].isin(leakage_vars)]

# Sortiert nach absoluter Effektgrösse
results_clean = results_clean.sort_values('abs_effect_size', ascending=False)

print(results_clean[results_clean['p_value'] < 0.05].head(30))


In [ ]:
# Features to select:

admin_features = [
    'NACCID',       # Unique Subject ID
    'VISITMO',      # Visit Monat
    'VISITYR',      # Visit Jahr  
    'VISITDAY',     # Visit Tag
    'NACCVNUM',     # Visit Nummer ← wie oft war er da
    'NACCNVST',     # Number of visits total
    'INVISITS',     # In-person visits
]

cognitive_features = [
     # Memory — sensitvster Bereich für MCI
    'LOGIMEM',      
    'MEMUNITS',     
    'CRAFTDVR',     
    'CRAFTDRE',     
    'CRAFTURS',     
    'CRAFTVRS',     

    # Executive Function
    'TRAILA', 'TRAILB',
    'DIGFORCT', 'DIGBACCT',
    'DIGBACLS',    
    
    # Global Cognition
    'NACCMMSE',
    'MOCATOTS',
    'NACCMOCA',     
    'CDRSUM',
    'CDRGLOB',
    
    # Language / Fluency
    'ANIMALS',     
    'VEG',          
    'BOSTON',
    'MOCAFLUE',
    
    # Weitere kognitive mit r > 0.15
    'MOCARECN',     
    'DIGFORSL',     
    'UDSVERTN',     
    'UDSVERLC',    
    'UDSVERFC',    
    'DIGBACLS',     
    'MOCAREPE',     
    'MOCASER7',     
    'MOCAABST',     
    'Group'
]

demographic_features = [
    'NACCAGE',      
    'EDUC',         
    'SEX',
    'HYPERTEN',     
    'DIABET',
    'HYPERCHO',
    'NACCBMI',
]

npi_features = [
    'NACCGDS',      # Geriatric Depression Scale
    'DEP',          # Depression
    'ANXIET',       # Anxiety  
    'BEAPATHY',     # Apathy ← früher Marker für MCI!
    'MEMPROB',      # Self-reported memory problems
]

features_of_interest = (
    admin_features +
    cognitive_features  + 
    demographic_features +
    npi_features
)

In [ ]:
data_feature_select = data_invest_with_groups[features_of_interest]

In [ ]:
display(data_feature_select.head())
print(set(data_feature_select.columns))

In [ ]:
print(set(data_biom.columns))

In [ ]:
df_nl_mci = data_feature_select[(data_feature_select['Group'] == 'NL to MCI') | (data_feature_select['Group'] == 'NL to NL')]


In [ ]:
data_biom['VISIT_DATE'] = pd.to_datetime(
    data_biom[['CSFLPYR', 'CSFLPMO', 'CSFLPDY']]
    .rename(columns={'CSFLPYR': 'year', 'CSFLPMO': 'month', 'CSFLPDY': 'day'}),
    errors='coerce'
)

df_nl_mci['VISIT_DATE'] = pd.to_datetime(
    df_nl_mci[['VISITYR', 'VISITMO', 'VISITDAY']]
    .rename(columns={'VISITYR': 'year', 'VISITMO': 'month', 'VISITDAY': 'day'}),
    errors='coerce'
)

In [ ]:
display(data_biom)

In [ ]:
display(df_nl_mci)
print(df_nl_mci['Group'].unique())

In [ ]:
df_nl_mci_sorted = df_nl_mci.sort_values('VISIT_DATE')
data_biom_sorted = data_biom.sort_values('VISIT_DATE')

df_gen_biom = pd.merge_asof(
    df_nl_mci_sorted,
    data_biom_sorted,
    on='VISIT_DATE',
    by='NACCID',
    direction='nearest',
    tolerance=pd.Timedelta('180 days')  # max 6 Monate Abstand
)

print(len(df_gen_biom.dropna()))
